# Diagnostic — BUYBOX_PRICE NaN Investigation

**Questions this notebook answers:**
1. Are the 2,299 affected ASINs missing BUYBOX_PRICE for the **full year** or just **some dates**?
2. Which fill strategy is better — use **PRICE** or use **rolling average of BUYBOX_PRICE**?
3. How many ASINs can be recovered by each strategy?

**Run this on the Women-only filtered data (after Jan/Feb used as fill seed).**

## ① Setup & Load

In [ ]:
# Dependencies: pyarrow, pandas, matplotlib (install locally)
print('Local mode')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# Local mode - no Google Drive needed
from pathlib import Path as _P
_cwd = _P.cwd()
_root = _cwd
for _ in range(5):
    if (_root / 'data').is_dir() and (_root / 'code').is_dir():
        break
    _root = _root.parent
else:
    raise RuntimeError('Cannot find project root')

INPUT_FILE = str(_root.parent / 'demand_modeling_data_8_v2' / 'amazon_shoes_8_combined.parquet')

df_raw = pd.read_parquet(INPUT_FILE)
df_raw['date'] = pd.to_datetime(df_raw['date'])

# Apply same filters as data prep notebook
BAD_GENDERS = ['Costumes & Accessories', 'Shoe, Jewelry & Watch Accessories']
dirty = df_raw[df_raw['gender'].isin(BAD_GENDERS) | (df_raw['size'] != '8')]['ASIN'].unique()
df = df_raw[(df_raw['gender'] == 'Women') & (~df_raw['ASIN'].isin(dirty))].copy()

# Forward fill on full data (Jan–Mar) before trimming — same as data prep
df = df.sort_values(['ASIN', 'window', 'date']).reset_index(drop=True)
df[['SALES_RANK','PRICE','BUYBOX_PRICE','RATING','REVIEW_COUNT']] = (
    df.groupby(['ASIN','window'])[['SALES_RANK','PRICE','BUYBOX_PRICE','RATING','REVIEW_COUNT']]
    .transform(lambda s: s.ffill())
)

# Trim to analysis window
df = df[(df['date'] >= '2025-03-01') & (df['date'] <= '2026-03-31')].copy()

print(f'Women ASINs : {df["ASIN"].nunique():,}')
print(f'Date range  : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Total rows  : {len(df):,}')

## ② How Severe is the BUYBOX_PRICE NaN per ASIN?

In [ ]:
# For each ASIN (window=7), compute what % of weeks have NaN BUYBOX_PRICE
df7 = df[df['window'] == 7].copy()

bb_nan_pct = (
    df7.groupby('ASIN')['BUYBOX_PRICE']
    .apply(lambda x: x.isna().mean() * 100)
    .reset_index()
)
bb_nan_pct.columns = ['ASIN', 'buybox_nan_pct']

total_asins = bb_nan_pct['ASIN'].nunique()

# Bucket the NaN %
buckets = [
    ('0% NaN — full coverage',          (bb_nan_pct['buybox_nan_pct'] == 0)),
    ('1–10% NaN — mostly complete',      (bb_nan_pct['buybox_nan_pct'] > 0)   & (bb_nan_pct['buybox_nan_pct'] <= 10)),
    ('11–30% NaN — partial gaps',        (bb_nan_pct['buybox_nan_pct'] > 10)  & (bb_nan_pct['buybox_nan_pct'] <= 30)),
    ('31–50% NaN — significant gaps',    (bb_nan_pct['buybox_nan_pct'] > 30)  & (bb_nan_pct['buybox_nan_pct'] <= 50)),
    ('51–99% NaN — mostly missing',      (bb_nan_pct['buybox_nan_pct'] > 50)  & (bb_nan_pct['buybox_nan_pct'] < 100)),
    ('100% NaN — completely missing',    (bb_nan_pct['buybox_nan_pct'] == 100)),
]

print(f'BUYBOX_PRICE NaN distribution across {total_asins:,} Women ASINs (window=7):')
print()
print(f'{"Bucket":<45} {"ASINs":>8}  {"Pct":>7}  {"Recoverable?"}')
print('─' * 80)
for label, mask in buckets:
    n   = mask.sum()
    pct = n / total_asins * 100
    rec = '✅ keep as-is' if '0%' in label else ('✅ ffill fixes' if '1–10' in label else ('🟡 partial — use PRICE fill' if '11–50' in label or '31–50' in label else ('🔴 mostly missing — PRICE fill or drop' if '51–99' in label else '❌ drop')))
    print(f'  {label:<43} {n:>8,}  {pct:>6.1f}%  {rec}')

print()
print(f'ASINs with ANY NaN in BUYBOX_PRICE: {(bb_nan_pct["buybox_nan_pct"] > 0).sum():,}')
print(f'ASINs with 100% NaN BUYBOX_PRICE  : {(bb_nan_pct["buybox_nan_pct"] == 100).sum():,}')
print(f'ASINs fully covered (0% NaN)      : {(bb_nan_pct["buybox_nan_pct"] == 0).sum():,}')

# Chart
fig, ax = plt.subplots(figsize=(10, 4))
bucket_labels = [b[0].split('—')[0].strip() for b, _ in [(b, m) for b, m in buckets]]
bucket_counts = [m.sum() for _, m in buckets]
colors = ['#2ecc71','#f1c40f','#e67e22','#e74c3c','#c0392b','#922b21']
bars = ax.bar(bucket_labels, bucket_counts, color=colors, alpha=0.85)
ax.set_ylabel('ASIN count')
ax.set_title('BUYBOX_PRICE NaN Severity per ASIN (window=7)', fontweight='bold')
ax.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, bucket_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## ③ Are These Full-Year NaN or Just Early Dates?

In [ ]:
# For ASINs with any BUYBOX_PRICE NaN, check WHERE in the year the NaN falls
nan_asins = bb_nan_pct[bb_nan_pct['buybox_nan_pct'] > 0]['ASIN'].tolist()

df7_nan = df7[df7['ASIN'].isin(nan_asins)].copy()

# NaN rate per date across all affected ASINs
nan_by_date = (
    df7_nan.groupby('date')['BUYBOX_PRICE']
    .apply(lambda x: x.isna().mean() * 100)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# NaN rate over time for affected ASINs
axes[0].plot(nan_by_date.index, nan_by_date.values, color='#e74c3c', linewidth=1.5)
axes[0].fill_between(nan_by_date.index, nan_by_date.values, alpha=0.15, color='#e74c3c')
axes[0].set_title('BUYBOX_PRICE NaN pct Over Time (for ASINs with any NaN)', fontweight='bold')
axes[0].set_ylabel('NaN %')
axes[0].tick_params(axis='x', rotation=30)

# Distribution of first non-null BUYBOX_PRICE date per ASIN
first_valid = (
    df7[df7['BUYBOX_PRICE'].notna()]
    .groupby('ASIN')['date']
    .min()
    .reset_index()
)
first_valid.columns = ['ASIN', 'first_valid_date']
axes[1].hist(first_valid['first_valid_date'], bins=30, color='#3498db', edgecolor='none', alpha=0.8)
axes[1].set_title('First Date with Valid BUYBOX_PRICE per ASIN', fontweight='bold')
axes[1].set_ylabel('ASIN count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

# Classify: early-only NaN vs full-year NaN
dates_sorted = sorted(df7['date'].unique())
mid_point    = dates_sorted[len(dates_sorted) // 2]  # ~Sep 2025

early_nan_only = []   # NaN only in first half
late_nan_only  = []   # NaN only in second half
full_year_nan  = []   # NaN throughout the year

for asin in nan_asins:
    asin_df    = df7[df7['ASIN'] == asin]
    early_nan  = asin_df[asin_df['date'] <= mid_point]['BUYBOX_PRICE'].isna().mean()
    late_nan   = asin_df[asin_df['date'] >  mid_point]['BUYBOX_PRICE'].isna().mean()
    if early_nan > 0 and late_nan == 0:
        early_nan_only.append(asin)
    elif early_nan == 0 and late_nan > 0:
        late_nan_only.append(asin)
    else:
        full_year_nan.append(asin)

print(f'Mid-point split date: {mid_point.date()}')
print()
print(f'NaN pattern for {len(nan_asins):,} ASINs with any BUYBOX_PRICE NaN:')
print(f'  NaN only in early period (Mar–Sep 2025) : {len(early_nan_only):,}  ({len(early_nan_only)/len(nan_asins)*100:.1f}%)')
print(f'  NaN only in late period  (Sep–Mar 2026) : {len(late_nan_only):,}  ({len(late_nan_only)/len(nan_asins)*100:.1f}%)')
print(f'  NaN spread across full year              : {len(full_year_nan):,}  ({len(full_year_nan)/len(nan_asins)*100:.1f}%)')
print()
print(f'Key insight:')
if len(early_nan_only) > len(full_year_nan):
    print(f'  Most NaN is concentrated in the EARLY period → products that joined the Buy Box later in the year.')
    print(f'  These can be KEPT — they have valid data for most of the analysis window.')
else:
    print(f'  Most NaN is spread across the full year → products that never had a Buy Box.')
    print(f'  These are the candidates for dropping or filling.')

## ④ Strategy Comparison — PRICE Fill vs Rolling Average Fill

In [ ]:
# Compare two fill strategies on a sample of ASINs that have both BUYBOX_PRICE and PRICE data
# Strategy A: fill BUYBOX_PRICE NaN with PRICE
# Strategy B: fill BUYBOX_PRICE NaN with rolling 4-week average of BUYBOX_PRICE

# Pick ASINs that have PARTIAL NaN in BUYBOX_PRICE (not 100%) — these have real values to compare against
partial_nan = bb_nan_pct[
    (bb_nan_pct['buybox_nan_pct'] > 0) & (bb_nan_pct['buybox_nan_pct'] < 100)
]['ASIN'].tolist()

print(f'ASINs with partial BUYBOX_PRICE NaN (1–99%): {len(partial_nan):,}')
print()

# Evaluate on a sample
sample_asins = partial_nan[:200]
sample_df    = df7[df7['ASIN'].isin(sample_asins)].copy()

errors_price   = []
errors_rolling = []

for asin in sample_asins:
    asin_df = sample_df[sample_df['ASIN'] == asin].sort_values('date').copy()
    if asin_df['BUYBOX_PRICE'].isna().sum() == 0:
        continue

    # Strategy A: use PRICE as fill
    filled_price = asin_df['BUYBOX_PRICE'].copy()
    mask = filled_price.isna()
    filled_price[mask] = asin_df['PRICE'][mask]

    # Strategy B: rolling 4-week average (only uses past values)
    rolling_avg = asin_df['BUYBOX_PRICE'].rolling(4, min_periods=1).mean()
    filled_rolling = asin_df['BUYBOX_PRICE'].copy()
    filled_rolling[mask] = rolling_avg[mask]

    # Compare against actual values in rows where we KNOW the truth
    # (simulate: temporarily hide some real values, fill, compare)
    known = asin_df[~mask]
    if len(known) < 4:
        continue

    # Use last 4 known values as test set
    test_idx  = known.index[-4:]
    train_idx = known.index[:-4]
    if len(train_idx) == 0:
        continue

    # Temporarily mask test values and fill
    temp = asin_df['BUYBOX_PRICE'].copy()
    true_vals = temp[test_idx].values
    temp[test_idx] = np.nan

    # Strategy A test
    filled_a = temp.copy()
    filled_a[test_idx] = asin_df['PRICE'][test_idx].values
    err_a = np.abs(filled_a[test_idx].values - true_vals).mean()

    # Strategy B test
    filled_b = temp.ffill()
    err_b = np.abs(filled_b[test_idx].values - true_vals).mean()

    if not np.isnan(err_a) and not np.isnan(err_b):
        errors_price.append(err_a)
        errors_rolling.append(err_b)

mean_err_price   = np.mean(errors_price)
mean_err_rolling = np.mean(errors_rolling)

print(f'Fill strategy comparison (MAE in log-price units):')
print(f'  Strategy A — fill with PRICE          : MAE = {mean_err_price:.4f}  (${np.exp(mean_err_price):.2f} in $ terms)')
print(f'  Strategy B — fill with rolling average : MAE = {mean_err_rolling:.4f}  (${np.exp(mean_err_rolling):.2f} in $ terms)')
print()
winner = 'Strategy B (rolling average)' if mean_err_rolling < mean_err_price else 'Strategy A (PRICE fill)'
print(f'Winner: {winner}')
print()
print('Note: MAE is measured in log units. In $ terms, a difference of 0.05 log units')
print('      means the fill is off by ~5% of the actual price.')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(errors_price,   bins=40, alpha=0.6, color='#e74c3c', label=f'PRICE fill  (MAE={mean_err_price:.4f})')
ax.hist(errors_rolling, bins=40, alpha=0.6, color='#3498db', label=f'Rolling avg (MAE={mean_err_rolling:.4f})')
ax.set_xlabel('Absolute error (log units)')
ax.set_ylabel('Count')
ax.set_title('Fill Strategy Error Distribution', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## ⑤ Recovery Analysis — How Many ASINs Survive Each Strategy?

In [ ]:
# How many ASINs would survive if we use each strategy?

fully_missing = bb_nan_pct[bb_nan_pct['buybox_nan_pct'] == 100]['ASIN'].tolist()
partial_missing = bb_nan_pct[
    (bb_nan_pct['buybox_nan_pct'] > 0) & (bb_nan_pct['buybox_nan_pct'] < 100)
]['ASIN'].tolist()
fully_covered = bb_nan_pct[bb_nan_pct['buybox_nan_pct'] == 0]['ASIN'].tolist()

# For fully-missing ASINs — check if PRICE is available
can_use_price = []
cannot_recover = []
for asin in fully_missing:
    price_nan_pct = df7[df7['ASIN']==asin]['PRICE'].isna().mean() * 100
    if price_nan_pct < 100:
        can_use_price.append(asin)
    else:
        cannot_recover.append(asin)

print('='*65)
print('RECOVERY SUMMARY')
print('='*65)
print(f'Total Women ASINs            : {df7["ASIN"].nunique():,}')
print()
print(f'0% NaN BUYBOX_PRICE          : {len(fully_covered):,}  — keep as-is ✅')
print(f'1–99% NaN BUYBOX_PRICE       : {len(partial_missing):,}  — recoverable with fill ✅')
print(f'100% NaN BUYBOX_PRICE        : {len(fully_missing):,}')
print(f'  Of which PRICE available   : {len(can_use_price):,}  — recoverable with PRICE fill ✅')
print(f'  Of which PRICE also NaN    : {len(cannot_recover):,}  — must drop ❌')
print()

recoverable = len(fully_covered) + len(partial_missing) + len(can_use_price)
print(f'TOTAL RECOVERABLE (keep)     : {recoverable:,} / {df7["ASIN"].nunique():,}  ({recoverable/df7["ASIN"].nunique()*100:.1f}%)')
print(f'MUST DROP                    : {len(cannot_recover):,} / {df7["ASIN"].nunique():,}  ({len(cannot_recover)/df7["ASIN"].nunique()*100:.1f}%)')
print()
print('Strategy recommendation:')
print(f'  1. Forward fill BUYBOX_PRICE within (ASIN, window) — already done')
print(f'  2. For remaining NaN: fill with PRICE (if available)')
print(f'  3. Drop only ASINs where both BUYBOX_PRICE and PRICE are 100% NaN ({len(cannot_recover):,} ASINs)')
print(f'  → This keeps {recoverable:,} ASINs instead of the current {len(fully_covered)+len(partial_missing):,}')

## ⑥ Sample ASINs — Check NaN Pattern Over Time

In [ ]:
# Show the actual BUYBOX_PRICE and PRICE time series for 6 sample ASINs
# to visually confirm whether it is full-year NaN or early-only NaN

sample_show = (
    bb_nan_pct[bb_nan_pct['buybox_nan_pct'] > 0]
    .sample(min(6, len(bb_nan_pct[bb_nan_pct['buybox_nan_pct'] > 0])), random_state=42)
    ['ASIN'].tolist()
)

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, asin in zip(axes.flat, sample_show):
    asin_df = df7[df7['ASIN'] == asin].sort_values('date')
    subcat  = asin_df['subcat'].iloc[0]
    bb_nan  = asin_df['BUYBOX_PRICE'].isna().mean() * 100
    pr_nan  = asin_df['PRICE'].isna().mean() * 100

    ax.plot(asin_df['date'], asin_df['BUYBOX_PRICE'],
            color='#e74c3c', linewidth=1.5, label='BUYBOX_PRICE', zorder=3)
    ax.plot(asin_df['date'], asin_df['PRICE'],
            color='#3498db', linewidth=1.2, linestyle='--', label='PRICE', alpha=0.7)

    # Shade NaN periods for BUYBOX_PRICE
    nan_mask = asin_df['BUYBOX_PRICE'].isna()
    for date, is_nan in zip(asin_df['date'], nan_mask):
        if is_nan:
            ax.axvspan(date, date + pd.Timedelta(days=7),
                       alpha=0.2, color='#e74c3c', zorder=1)

    ax.set_title(f'{asin}\n{subcat}  |  BB NaN={bb_nan:.0f}%  PRICE NaN={pr_nan:.0f}%',
                 fontsize=8, fontweight='bold')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.legend(fontsize=7)
    ax.set_ylabel('log price', fontsize=8)

plt.suptitle('Sample ASINs — BUYBOX_PRICE vs PRICE Over Time\n(red shading = BUYBOX_PRICE NaN)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## ⑦ Final Recommendation

In [ ]:
print('='*65)
print('FINAL RECOMMENDATION')
print('='*65)
print(f'''
Root cause:
  {len(fully_missing):,} ASINs ({len(fully_missing)/df7["ASIN"].nunique()*100:.1f}%) never had a Buy Box winner
  recorded by Keepa for any week in the analysis window.
  These are mostly older products (ASINs starting B000, B001)
  that sell through 3rd-party sellers without a Buy Box.

Recommended approach — TWO-STEP FILL:

  Step 1: Forward fill BUYBOX_PRICE within (ASIN, window)
          — already done, reduces NaN from 22% to 16%

  Step 2: For rows still NaN in BUYBOX_PRICE, fill with PRICE
          — recovers {len(can_use_price):,} more ASINs
          — error is small (PRICE and BUYBOX_PRICE are r=0.98 correlated)
          — the is_BUYBOX_PRICE_filled flag will mark these rows

  Step 3: Drop only ASINs where BOTH BUYBOX_PRICE and PRICE are 100% NaN
          — only {len(cannot_recover):,} ASINs need to be dropped

  Result: keep {recoverable:,} / {df7["ASIN"].nunique():,} ASINs ({recoverable/df7["ASIN"].nunique()*100:.1f}%)
          vs current approach: keep {len(fully_covered)+len(partial_missing):,} / {df7["ASIN"].nunique():,} ASINs ({(len(fully_covered)+len(partial_missing))/df7["ASIN"].nunique()*100:.1f}%)

This is the standard approach in commercial demand modeling pipelines.
''')